# RQ2 - Poisoning the base learner

Label-flipping against the full coupled adaptive system. Experiments include: 
1. baselines
2. the suppression-zone attack
3. the encoder/exemplar ablations.


In [ ]:
import os, sys, copy, pickle
from typing import Callable

sys.path.insert(0, os.path.abspath('../..'))

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt

from src.experiments.contrastive_ncm import (
    load_cicids2017,
    prepare_temporal_split,
    BaseLearnerMLP,
    train_base_learner,
    eval_base_learner,
    ConceptBuffer,
    BatchResult,
    StaticDStaticM,
    StaticDAdaptiveM,
    AdaptiveDAdaptiveM,
    run_stream,
    initial_training,
    apply_paper_style,
    save_fig,
    save_latex,
    agg,
    DAY_NAMES_DEFAULT,
)
from src.detectors.contrastive_ncm import ContrastiveNCMDetector

RANDOM_SEED = 42
torch.manual_seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
torch.set_float32_matmul_precision('high')

DATA_DIR = os.path.abspath('../../data/CICIDS2017_Engelen')
FIGURE_DIR = os.path.abspath('./figures')
TABLE_DIR = os.path.abspath('./tables')

NOVEL_CLASSES = ['DoS', 'PortScan']
CONCEPT_BATCH = 256
BATCH_SIZE = 1000
POISON_FRACS = [0.0, 0.05, 0.15, 0.20, 0.25, 0.30, 0.35]

DD_RETRAIN_EPOCHS = 300
DD_RETRAIN_LR = 1e-4
DD_EVAL_POOL_N = 2000

N_SEEDS = 3
SEEDS = [RANDOM_SEED + 1000 + i for i in range(N_SEEDS)]

M_HIDDEN = (256, 128, 64)

apply_paper_style()

DAY_NAMES = DAY_NAMES_DEFAULT

from src.experiments.contrastive_ncm import set_progress, set_stream_verbose
PROGRESS = False
set_progress(PROGRESS)
set_stream_verbose(PROGRESS)

/home/azureuser/.local/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Load and preprocess


In [ ]:
print('Loading CICIDS2017...')
data = load_cicids2017(DATA_DIR)

print(f'  Rows: {len(data.X):,}  Features: {data.input_dim}')
for cls in data.le_full.classes_:
    n = (data.y_str == cls).sum()
    tag = ' <- NOVEL' if cls in NOVEL_CLASSES else ''
    print(f'    {cls:<22} {n:>8,}  ({100*n/len(data.y_str):.1f}%){tag}')

print('\nBuilding temporal split...')
split = prepare_temporal_split(
    data,
    novel_classes=NOVEL_CLASSES,
    per_class_metric_names=NOVEL_CLASSES,
    random_seed=RANDOM_SEED,
)
print("\n")
for didx, dname in enumerate(DAY_NAMES):
    mask = split.d_stream == didx
    if not mask.any():
        continue
    
    n_nov = np.isin(split.y_stream[mask], split.novel_ids).sum()
    print(f'  {dname}: {mask.sum():>7,} ({100*n_nov/mask.sum():.1f}% novel)')


Loading CICIDS2017...


  Rows: 2,100,814  Features: 77
    BENIGN                 1,666,837  (79.3%)
    Bot                         738  (0.0%)
    DDoS                     95,123  (4.5%)
    DoS                     171,779  (8.2%) <- NOVEL
    Heartbleed                   11  (0.0%)
    Infiltration                 32  (0.0%)
    Patator                   6,953  (0.3%)
    PortScan                159,151  (7.6%) <- NOVEL
    Web Attack                  190  (0.0%)



  Mon:  92,083 (0.0% novel)
  Tue:  79,181 (0.0% novel)
  Wed: 249,496 (67.8% novel)
  Thu:  89,878 (0.0% novel)
  Fri: 252,763 (62.0% novel)


## Initial training


In [ ]:
print('Initial training...')
init = initial_training(
    X_tr=split.X_tr,
    y_tr_mc=split.y_tr_mc,
    y_tr_bin=split.y_tr_bin,
    n_known=split.n_known,
    input_dim=data.input_dim,
    device=DEVICE,
    random_seed=RANDOM_SEED,
)

print(f'drift_threshold     = {init.drift_threshold:.4f}')
print(f'concept_threshold T = {init.concept_threshold:.4f}')

m0 = eval_base_learner(
    init.base_learner,
    split.X_eval,
    split.y_eval_bin,
    split.novel_mask_eval,
    split.per_class_masks,
)

print(f'\nPost-initial M  accuracy={m0["accuracy"]:.3f}  f1={m0["f1"]:.3f}')
print(f'  FNR novel = {m0["fnr_novel"]:.3f}')

for k, v in sorted(m0.items()):
    if k.startswith('fnr_') and k != 'fnr_novel':
        print(f'{k:18s} = {v:.3f}')


Initial training...


    [Contrastive 20/300] loss=0.1477


    [Contrastive 40/300] loss=0.0835


    [Contrastive 60/300] loss=0.0492


    [Contrastive 80/300] loss=0.0359


...

    [M 40/100] loss=0.0004


    [M 60/100] loss=0.0002


    [M 80/100] loss=0.0002


    [M 100/100] loss=0.0005


drift_threshold     = 1.3339
concept_threshold T = 37.8378

Post-initial M  accuracy=0.531  f1=0.207
FNR novel = 0.937
fnr_dos            = 0.881
fnr_portscan       = 0.999


## Detector evaluation pool


In [ ]:
_rng_pool = np.random.default_rng(RANDOM_SEED + 7)
pool_idx = _rng_pool.choice(len(split.novel_pool), DD_EVAL_POOL_N, replace=False)
X_dd_eval_pool = split.novel_pool[pool_idx]
print(f'DD eval pool: {len(X_dd_eval_pool):,} samples')

with torch.no_grad():
    _, _is_dr, _ = init.detector.detect(torch.from_numpy(X_dd_eval_pool))

print(f'Initial DD drift rate on pool: {100*_is_dr.float().mean():.1f}%')


DD eval pool: 2,000 samples
Initial DD drift rate on pool: 30.6%


## System builders


In [ ]:
def _fresh_detector():
    """Reconstruct a fresh detector from the cached initial state."""
    det = ContrastiveNCMDetector(
        input_dim=data.input_dim,
        hidden_dim=64,
        latent_dim=32,
        temperature=0.1,
        device=DEVICE,
    )
    
    det.autoencoder.load_state_dict(copy.deepcopy(init.dd_init_ae))
    det.ncm = copy.deepcopy(init.dd_init_ncm)
    det.drift_threshold = init.drift_threshold
    det.concept_threshold = init.concept_threshold
    det.num_classes = split.n_known
    det._reset_drift_buffer()

    return det

def _fresh_m():
    ml = BaseLearnerMLP(data.input_dim, M_HIDDEN).to(DEVICE)
    ml.load_state_dict(copy.deepcopy(init.m_init_state))
    return ml


def fresh_full_system():
    """AdaptiveDAdaptiveM - both DD encoder and M retrain on concept discovery."""
    return AdaptiveDAdaptiveM(
        detector=_fresh_detector(),
        base_learner=_fresh_m(),
        X_exemplars=init.X_exemplars,
        y_exemplars=init.y_exemplars,
        y_exemplars_mc=init.y_exemplars_mc,
        m_init_state=init.m_init_state,
        X_eval=split.X_eval,
        y_eval_bin=split.y_eval_bin,
        novel_mask=split.novel_mask_eval,
        X_dd_eval_pool=X_dd_eval_pool,
        per_class_masks=split.per_class_masks,
        concept_batch=CONCEPT_BATCH,
        dd_retrain_epochs=DD_RETRAIN_EPOCHS,
        dd_retrain_lr=DD_RETRAIN_LR,
        X_cal=init.X_cal,
        y_cal_mc=init.y_cal_mc,
    )

def fresh_static_dd_system():
    """StaticDAdaptiveM - DD encoder frozen, M retrains. Used for ablation only."""
    return StaticDAdaptiveM(
        detector=_fresh_detector(),
        base_learner=_fresh_m(),
        X_exemplars=init.X_exemplars,
        y_exemplars=init.y_exemplars,
        m_init_state=init.m_init_state,
        X_eval=split.X_eval,
        y_eval_bin=split.y_eval_bin,
        novel_mask=split.novel_mask_eval,
        per_class_masks=split.per_class_masks,
        concept_batch=CONCEPT_BATCH,
    )

def fresh_static_baseline():
    """StaticDStaticM - M never retrains. Pre-deployment reference line."""
    return StaticDStaticM(
        _fresh_m(),
        split.X_eval,
        split.y_eval_bin,
        split.novel_mask_eval,
        split.per_class_masks,
    )


## Baselines


In [ ]:
print('Running StaticDStaticM baseline...')
static_sys = fresh_static_baseline()
static_results = run_stream(
    static_sys,
    X_stream=split.X_stream,
    y_stream_bin=split.y_stream_bin,
    d_stream=split.d_stream,
    novel_pool=split.novel_pool,
    poison_frac=0.0,
    rng_seed=SEEDS[0],
    batch_size=BATCH_SIZE,
)

print('Running StaticDAdaptiveM baseline (ablation reference)...')
static_dd_sys = fresh_static_dd_system()
static_dd_results = run_stream(
    static_dd_sys,
    X_stream=split.X_stream,
    y_stream_bin=split.y_stream_bin,
    d_stream=split.d_stream,
    novel_pool=split.novel_pool,
    poison_frac=0.0,
    rng_seed=SEEDS[0],
    batch_size=BATCH_SIZE,
)

print(f'Total M retrains: {static_dd_sys.retrain_count}')

print('Running AdaptiveDAdaptiveM baseline (primary system)...')
full_sys = fresh_full_system()
full_results = run_stream(
    full_sys,
    X_stream=split.X_stream,
    y_stream_bin=split.y_stream_bin,
    d_stream=split.d_stream,
    novel_pool=split.novel_pool,
    poison_frac=0.0,
    rng_seed=SEEDS[0],
    batch_size=BATCH_SIZE,
)

print(f'Total M retrains : {full_sys.retrain_count}')
print(f'Total DD retrains: {full_sys.n_dd_retrains}')
print(f'Final NCM prototypes: {full_sys.detector.ncm.num_classes} ' f'(initial {split.n_known})')

baseline_rows = []
prev_dd_retrains = 0
for didx, dname in enumerate(DAY_NAMES):
    s = [r for r in static_results if r.day_idx == didx]
    sd = [r for r in static_dd_results if r.day_idx == didx]
    f = [r for r in full_results if r.day_idx == didx]
    if not s:
        continue
    dd_ret_now = max(r.extras['n_dd_retrains'] for r in f) if f else prev_dd_retrains
    dd_ret_today = dd_ret_now - prev_dd_retrains
    prev_dd_retrains = dd_ret_now
    baseline_rows.append(
        {
            'Day': dname,
            'F1 bench (StaticStatic)': float(np.mean([r.f1 for r in s])),
            'F1 stream (StaticStatic)': float(np.mean([r.extras['f1_stream'] for r in s])),
            'F1 bench (StaticDD)': float(np.mean([r.f1 for r in sd])),
            'F1 stream (StaticDD)': float(np.mean([r.extras['f1_stream'] for r in sd])),
            'F1 bench (AdaptiveDD)': float(np.mean([r.f1 for r in f])),
            'F1 stream (AdaptiveDD)': float(np.mean([r.extras['f1_stream'] for r in f])),
            'DD drift pool': float(np.mean([r.extras['dd_drift_rate_pool'] for r in f])),
            'M retrains (Adapt)': int(sum(r.retrain_fired for r in f)),
            'DD retrains (Adapt)': int(dd_ret_today),
        }
    )
baseline_df = pd.DataFrame(baseline_rows).set_index('Day')
print("\n")

display(baseline_df.round(3))
save_latex(
    baseline_df.round(3),
    'baseline_adaptation.tex',
    TABLE_DIR,
    caption=(
        'Baseline (no attack) per-day metrics for the static reference, '
        'static-encoder ablation, and the full system (AdaptiveDAdaptiveM). '
        'F1 bench measures M against the fixed novel-class-aware benchmark; '
        'F1 stream measures M against the day\'s actual stream traffic.'
    ),
    label='tab:baseline-adaptation',
)


Running StaticDStaticM baseline...


Running StaticDAdaptiveM baseline (ablation reference)...


    [M 20/100] loss=0.0031


    [M 40/100] loss=0.0009


    [M 60/100] loss=0.0009


...

    [Contrastive 200/300] loss=1.5011
    [Contrastive 220/300] loss=1.2110


    [Contrastive 240/300] loss=1.0999
    [Contrastive 260/300] loss=1.0566


    [Contrastive 280/300] loss=0.9608
    [Contrastive 300/300] loss=0.8692


Total M retrains : 1
Total DD retrains: 1
Final NCM prototypes: 8 (initial 7)



,F1 bench (StaticStatic),F1 stream (StaticStatic),F1 bench (StaticDD),F1 stream (StaticDD),F1 bench (AdaptiveDD),F1 stream (AdaptiveDD),DD drift pool,M retrains (Adapt),DD retrains (Adapt)
Day,,,,,,,,,
Mon,0.207,0.000,0.207,0.000,0.207,0.000,0.306,0,0
Tue,0.207,0.682,0.207,0.682,0.207,0.682,0.306,0,0
Wed,0.207,0.219,0.344,0.405,0.346,0.402,0.228,1,1
Thu,0.207,0.404,0.698,0.283,0.710,0.229,0.026,0,0
Fri,0.207,0.170,0.698,0.163,0.710,0.192,0.026,0,0


In [ ]:
print("\n=== Baseline FNR for novel classes (StaticDStaticM) ===")
print(f'{"Day":<5} {"FNR(DoS)":>10} {"FNR(PortScan)":>14}')
for didx, dname in enumerate(DAY_NAMES):
    a = [r for r in static_results if r.day_idx == didx]
    if not a:
        continue
    fdos = np.nanmean([r.extras.get("fnr_dos", np.nan) for r in a])
    fps = np.nanmean([r.extras.get("fnr_portscan", np.nan) for r in a])
    print(f"{dname:<5} {fdos:>10.3f} {fps:>14.3f}")

print("\n=== Baseline FNR for novel classes (AdaptiveDAdaptiveM) ===")
print(f'{"Day":<5} {"FNR(DoS)":>10} {"FNR(PortScan)":>14}')
for didx, dname in enumerate(DAY_NAMES):
    a = [r for r in full_results if r.day_idx == didx]
    if not a:
        continue
    fdos = np.nanmean([r.extras.get("fnr_dos", np.nan) for r in a])
    fps = np.nanmean([r.extras.get("fnr_portscan", np.nan) for r in a])
    print(f"{dname:<5} {fdos:>10.3f} {fps:>14.3f}")


In [ ]:
fri = [r for r in full_results if r.day_idx == 4]
df = pd.DataFrame(
    [
        {
            "batch": r.batch_idx,
            "buf": r.buf_size,
            "n_attack": r.extras["n_attack_in_buf"],
            "attack_frac": r.extras["attack_frac"],
            "allow_fire": r.extras["allow_fire"],
            "delta_norm": r.extras["delta_norm"],
            "T": r.extras["concept_T"],
        }
        for r in fri
    ]
)
print(df.to_string(index=False))

T = df["T"].iloc[-1]
print(f"\nmax delta_norm (Fri): {df.delta_norm.max():.2f}   (T = {T:.2f})")
print(f"max n_attack        : {df.n_attack.max()}   (count floor 500)")
print(f"max attack_frac     : {df.attack_frac.max():.3f} (frac floor 0.30)")

## Adversarial pool


In [ ]:
print(f'Novel pool: {len(split.novel_pool):,} samples')
_sys_tmp = fresh_full_system()
_samp = split.novel_pool[
    np.random.default_rng(0).choice(
        len(split.novel_pool), min(3000, len(split.novel_pool)), replace=False
    )
]

with torch.no_grad():
    _, _is_dr, _ = _sys_tmp.detector.detect(torch.from_numpy(_samp))
    
print(f'DD drift rate on novel pool (initial encoder): {100*_is_dr.float().mean():.1f}%')


Novel pool: 330,930 samples
DD drift rate on novel pool (initial encoder): 29.0%


In [ ]:
y_pool_full = data.y_all[np.isin(data.y_all, split.novel_ids)]
dos_id = int(data.le_full.transform(['DoS'])[0])
portscan_id = int(data.le_full.transform(['PortScan'])[0])
mask_dos = y_pool_full == dos_id
mask_ps = y_pool_full == portscan_id

print(f'Novel pool composition: DoS={mask_dos.sum():,}  PortScan={mask_ps.sum():,}')

rng = np.random.default_rng(7)
sub_dos = rng.choice(np.where(mask_dos)[0], min(5000, int(mask_dos.sum())), replace=False)
sub_ps = rng.choice(np.where(mask_ps)[0], min(5000, int(mask_ps.sum())), replace=False)
X_dos = split.novel_pool[sub_dos]
X_ps = split.novel_pool[sub_ps]

@torch.no_grad()
def drift_rate(det, X):
    _, is_drift, _ = det.detect(torch.from_numpy(X))
    return float(is_drift.float().mean())

init_det = copy.deepcopy(init.detector)
init_det.autoencoder.load_state_dict(copy.deepcopy(init.dd_init_ae))
init_det.ncm = copy.deepcopy(init.dd_init_ncm)
init_det.drift_threshold = init.drift_threshold
init_det.concept_threshold = init.concept_threshold
init_det.num_classes = split.n_known
init_det._reset_drift_buffer()

post_det = full_sys.detector

print("\n")
print('=== Per-class drift rate on the novel pool ===')
print(f'{"":24s}  {"initial enc.":>14s}   {"post-retrain":>14s}')
print(
    f'{"DoS":22s}  {100*drift_rate(init_det, X_dos):>13.1f}%   {100*drift_rate(post_det, X_dos):>13.1f}%'
)
print(
    f'{"PortScan":22s}  {100*drift_rate(init_det, X_ps):>13.1f}%   {100*drift_rate(post_det, X_ps):>13.1f}%'
)


Novel pool composition: DoS=171,779  PortScan=159,151

=== Per-class drift rate on the novel pool ===
                            initial enc.     post-retrain
DoS                              54.1%             2.8%
PortScan                          2.5%             1.4%


## Label-flipping attack


In [ ]:
attack_results = {p: [] for p in POISON_FRACS}
m_retrain_counts = {p: [] for p in POISON_FRACS}
dd_retrain_counts = {p: [] for p in POISON_FRACS}

for pfrac in POISON_FRACS:
    fri_f1, fri_dos, fri_ps, fri_drift, n_dd_ret, n_m_ret = [], [], [], [], [], []
    for seed in SEEDS:
        system = fresh_full_system()
        results = run_stream(
            system,
            X_stream=split.X_stream,
            y_stream_bin=split.y_stream_bin,
            d_stream=split.d_stream,
            novel_pool=split.novel_pool,
            poison_frac=pfrac,
            rng_seed=seed,
            batch_size=BATCH_SIZE,
        )

        attack_results[pfrac].append(results)
        m_retrain_counts[pfrac].append(system.retrain_count)
        dd_retrain_counts[pfrac].append(system.n_dd_retrains)

        fri = [r for r in results if r.day_idx == 4]
        if fri:
            fri_f1.append(np.mean([r.f1 for r in fri]))
            fri_dos.append(np.mean([r.extras['fnr_dos'] for r in fri]))
            fri_ps.append(np.mean([r.extras['fnr_portscan'] for r in fri]))
            fri_drift.append(np.mean([r.extras['dd_drift_rate_pool'] for r in fri]))
            n_m_ret.append(system.retrain_count)
            n_dd_ret.append(system.n_dd_retrains)

    print(
        f'p={pfrac:>5.0%} | Fri F1={np.mean(fri_f1):.3f}\xb1{np.std(fri_f1):.3f}  '
        f'FNR_DoS={np.mean(fri_dos):.3f}\xb1{np.std(fri_dos):.3f}  '
        f'FNR_PS={np.mean(fri_ps):.3f}\xb1{np.std(fri_ps):.3f}  '
        f'DD drift={np.mean(fri_drift):.3f}\xb1{np.std(fri_drift):.3f}  '
        f'M-retr={np.mean(n_m_ret):.0f}  DD-retr={np.mean(n_dd_ret):.1f}'
    )


    [M 20/100] loss=0.0031


    [M 40/100] loss=0.0010


    [M 60/100] loss=0.0010


    [M 80/100] loss=0.0004


    [M 100/100] loss=0.0004
    [Contrastive 20/300] loss=11.3253


...

    [Contrastive 160/300] loss=1.7114
    [Contrastive 180/300] loss=1.6947


    [Contrastive 200/300] loss=1.4122
    [Contrastive 220/300] loss=1.2936


    [Contrastive 240/300] loss=1.2525
    [Contrastive 260/300] loss=1.1464


    [Contrastive 280/300] loss=1.1151
    [Contrastive 300/300] loss=0.8778


p=  35% | Fri F1=0.702±0.001  FNR_DoS=0.006±0.001  FNR_PS=1.000±0.000  DD drift=0.101±0.103  M-retr=1  DD-retr=1.0


## Friday end-of-stream summary


In [ ]:
def per_day_metric(seed_runs, day_idx, attr_path):
    out = []
    for sr in seed_runs:
        day = [r for r in sr if r.day_idx == day_idx]
        if not day:
            continue
        if attr_path.startswith('extras.'):
            key = attr_path.split('.', 1)[1]
            out.append(float(np.mean([r.extras[key] for r in day])))
        else:
            out.append(float(np.mean([getattr(r, attr_path) for r in day])))
    return out


fri_rows = []
fri_rows.append(
    {
        'System': 'StaticDStaticM',
        'Poison %': 'ref',
        'Fri F1': agg(per_day_metric([static_results], 4, 'f1')),
        'Fri FNR DoS': agg(per_day_metric([static_results], 4, 'extras.fnr_dos')),
        'Fri FNR PS': agg(per_day_metric([static_results], 4, 'extras.fnr_portscan')),
        'M retrains': '0',
        'DD retrains': '0',
    }
)

fri_rows.append(
    {
        'System': 'AdaptiveDD p=0',
        'Poison %': '0%',
        'Fri F1': agg(per_day_metric([full_results], 4, 'f1')),
        'Fri FNR DoS': agg(per_day_metric([full_results], 4, 'extras.fnr_dos')),
        'Fri FNR PS': agg(per_day_metric([full_results], 4, 'extras.fnr_portscan')),
        'M retrains': f'{full_sys.retrain_count}',
        'DD retrains': f'{full_sys.n_dd_retrains}',
    }
)

for pfrac in POISON_FRACS[1:]:
    fri_rows.append(
        {
            'System': f'AdaptiveDD p={pfrac:.0%}',
            'Poison %': f'{pfrac:.0%}',
            'Fri F1': agg(per_day_metric(attack_results[pfrac], 4, 'f1')),
            'Fri FNR DoS': agg(per_day_metric(attack_results[pfrac], 4, 'extras.fnr_dos')),
            'Fri FNR PS': agg(per_day_metric(attack_results[pfrac], 4, 'extras.fnr_portscan')),
            'M retrains': f'{np.mean(m_retrain_counts[pfrac]):.0f}'
            f'\xb1{np.std(m_retrain_counts[pfrac]):.0f}',
            'DD retrains': f'{np.mean(dd_retrain_counts[pfrac]):.1f}'
            f'\xb1{np.std(dd_retrain_counts[pfrac]):.1f}',
        }
    )
fri_df = pd.DataFrame(fri_rows).set_index('System')

display(fri_df)
save_latex(
    fri_df,
    'attack_fri_summary.tex',
    TABLE_DIR,
    caption=(
        f'Friday end-of-stream attack effect on the full coupled adaptive IDS '
        f'(AdaptiveDAdaptiveM). Values are mean $\\pm$ std across {N_SEEDS} seeds.'
    ),
    label='tab:attack-fri-summary',
)


,Poison %,Fri F1,Fri FNR DoS,Fri FNR PS,M retrains,DD retrains
System,,,,,,
StaticDStaticM,ref,0.207 ± 0.000,0.881 ± 0.000,0.999 ± 0.000,0,0
AdaptiveDD p=0,0%,0.710 ± 0.000,0.008 ± 0.000,0.983 ± 0.000,1,1
AdaptiveDD p=5%,5%,0.726 ± 0.034,0.014 ± 0.006,0.903 ± 0.137,1±0,1.3±0.5
AdaptiveDD p=15%,15%,0.820 ± 0.053,0.030 ± 0.003,0.529 ± 0.205,2±0,2.0±0.0
AdaptiveDD p=20%,20%,0.821 ± 0.053,0.033 ± 0.002,0.508 ± 0.175,2±0,2.0±0.0
AdaptiveDD p=25%,25%,0.207 ± 0.000,0.881 ± 0.000,0.999 ± 0.000,0±0,0.0±0.0
AdaptiveDD p=30%,30%,0.680 ± 0.006,0.055 ± 0.011,1.000 ± 0.000,1±0,1.0±0.0
AdaptiveDD p=35%,35%,0.702 ± 0.001,0.006 ± 0.001,1.000 ± 0.000,1±0,1.0±0.0


## Ablation: static-encoder posture


In [ ]:
static_dd_attack_results = {p: [] for p in POISON_FRACS}
static_dd_m_retrains = {p: [] for p in POISON_FRACS}

for pfrac in POISON_FRACS:
    fri_f1, fri_dos, n_ret = [], [], []
    for seed in SEEDS:
        system = fresh_static_dd_system()
        results = run_stream(
            system,
            X_stream=split.X_stream,
            y_stream_bin=split.y_stream_bin,
            d_stream=split.d_stream,
            novel_pool=split.novel_pool,
            poison_frac=pfrac,
            rng_seed=seed,
            batch_size=BATCH_SIZE,
        )

        static_dd_attack_results[pfrac].append(results)
        static_dd_m_retrains[pfrac].append(system.retrain_count)

        fri = [r for r in results if r.day_idx == 4]
        if fri:
            fri_f1.append(np.mean([r.f1 for r in fri]))
            fri_dos.append(np.mean([r.extras['fnr_dos'] for r in fri]))
            n_ret.append(system.retrain_count)

    print(
        f'p={pfrac:>5.0%} | Fri F1={np.mean(fri_f1):.3f}\xb1{np.std(fri_f1):.3f}  '
        f'FNR_DoS={np.mean(fri_dos):.3f}\xb1{np.std(fri_dos):.3f}  '
        f'M-retr={np.mean(n_ret):.0f}\xb1{np.std(n_ret):.0f}'
    )


    [M 20/100] loss=0.0024


    [M 40/100] loss=0.0009


    [M 60/100] loss=0.0009


    [M 80/100] loss=0.0007


    [M 100/100] loss=0.0005
    [M 20/100] loss=0.0559


...

    [M 40/100] loss=0.5007


    [M 60/100] loss=0.5001


    [M 80/100] loss=0.5000


    [M 100/100] loss=0.4993


p=  35% | Fri F1=0.699±0.003  FNR_DoS=0.005±0.001  M-retr=1±0


## Static vs adaptive encoder


In [ ]:
ablation_rows = []
for pfrac in POISON_FRACS:
    full_f1 = per_day_metric(attack_results[pfrac], 4, 'f1')
    sdd_f1 = per_day_metric(static_dd_attack_results[pfrac], 4, 'f1')
    full_dos = per_day_metric(attack_results[pfrac], 4, 'extras.fnr_dos')
    sdd_dos = per_day_metric(static_dd_attack_results[pfrac], 4, 'extras.fnr_dos')

    if not full_f1 or not sdd_f1:
        continue

    ablation_rows.append(
        {
            'Poison %': f'{pfrac:.0%}',
            'Fri F1 (AdaptiveDD)': agg(full_f1),
            'Fri F1 (StaticDD)': agg(sdd_f1),
            'Fri FNR DoS (AdaptiveDD)': agg(full_dos),
            'Fri FNR DoS (StaticDD)': agg(sdd_dos),
        }
    )

ablation_df = pd.DataFrame(ablation_rows).set_index('Poison %')

display(ablation_df)
save_latex(
    ablation_df,
    'ablation_encoder_adaptation.tex',
    TABLE_DIR,
    caption=(
        'Ablation: full system (AdaptiveDD) vs static-encoder posture (StaticDD) '
        'under the same label-flipping attack. Friday end-of-stream metrics, '
        f'mean $\\pm$ std across {N_SEEDS} seeds.'
    ),
    label='tab:ablation-encoder',
)


,Fri F1 (AdaptiveDD),Fri F1 (StaticDD),Fri FNR DoS (AdaptiveDD),Fri FNR DoS (StaticDD)
Poison %,,,,
0%,0.807 ± 0.103,0.700 ± 0.001,0.019 ± 0.001,0.018 ± 0.002
5%,0.726 ± 0.034,0.651 ± 0.074,0.014 ± 0.006,0.118 ± 0.161
15%,0.820 ± 0.053,0.703 ± 0.002,0.030 ± 0.003,0.005 ± 0.001
20%,0.821 ± 0.053,0.704 ± 0.001,0.033 ± 0.002,0.004 ± 0.001
25%,0.207 ± 0.000,0.207 ± 0.000,0.881 ± 0.000,0.881 ± 0.000
30%,0.680 ± 0.006,0.306 ± 0.048,0.055 ± 0.011,0.740 ± 0.067
35%,0.702 ± 0.001,0.699 ± 0.003,0.006 ± 0.001,0.005 ± 0.001
